# 04 — Data augmentation and train/validation/test datasets

Builds three separate `tf.data.Dataset` objects from the shared manifest. Random augmentation is applied only to training images; validation and test are deterministic.

In [ ]:
import tensorflow as tf
from pathlib import Path
import pandas as pd
from tensorflow.keras.applications.resnet50 import preprocess_input

SEED=42; IMAGE_SIZE=(224,224); BATCH_SIZE=32
ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name=="04DataAugmentation" else ROOT
manifest=pd.read_csv(ROOT/"results"/"preprocessing"/"split_manifest.csv")
classes=sorted(manifest["class"].unique()); class_to_id={c:i for i,c in enumerate(classes)}

augment=tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.10, seed=SEED),
    tf.keras.layers.RandomZoom(0.10, seed=SEED),
], name="training_augmentation")

def load(path,label,training=False):
    image=tf.io.read_file(path); image=tf.image.decode_image(image,channels=3,expand_animations=False)
    image.set_shape([None,None,3]); image=tf.image.resize(image,IMAGE_SIZE); image=tf.cast(image,tf.float32)
    if training: image=augment(image, training=True)
    image=preprocess_input(image)
    return image,label

def make_dataset(split, training=False):
    part=manifest[manifest.split==split]
    ds=tf.data.Dataset.from_tensor_slices((part.path.values,part["class"].map(class_to_id).values))
    if training: ds=ds.shuffle(len(part),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(lambda p,y: load(p,y,training),num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_dataset("train",training=True)
validation_ds=make_dataset("validation",training=False)
test_ds=make_dataset("test",training=False)
print(train_ds, validation_ds, test_ds)


Train: shuffled and augmented. Validation: no random augmentation, used for model selection and early stopping. Test: no random augmentation, reserved for final evaluation. All three use the same ResNet50 `preprocess_input` normalization after resizing.